Read satellite data


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

GIMMS_PHENOLOGY_DIR = "../../data/satellite_data/images/PKU-GIMMS/phenology"
GIMMS_PHENOLOGY_PATTERN = "GIMMS_Phenology_SnowFilter_Forest1114_{year}.tif"

def load_gimms_snowfilter_sos_eos(df, years, phenology_dir=GIMMS_PHENOLOGY_DIR):
        coords = list(zip(df["longitude"].values, df["latitude"].values))
    n = len(coords)
    sample_year = next(
        y for y in years
        if os.path.exists(os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=y)))
    )
    with rasterio.open(os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=sample_year))) as src:
        transform = src.transform
        height, width = src.height, src.width

    rows = np.empty(n, dtype=np.int32)
    cols = np.empty(n, dtype=np.int32)
    for i, (lon, lat) in enumerate(coords):
        r, c = rasterio.transform.rowcol(transform, lon, lat)
        rows[i], cols[i] = r, c

    valid_rc = (rows >= 0) & (cols >= 0) & (rows < height) & (cols < width)

    for year in years:
        fp = os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=year))
        if not os.path.exists(fp):
            print(f"  missing phenology file: {fp}", flush=True)
            df[f"sos_{year}"] = np.nan
            df[f"eos_{year}"] = np.nan
            continue
        with rasterio.open(fp) as src:
            sos_band = src.read(1)
            eos_band = src.read(2)
        sos = np.full(n, np.nan, dtype=np.float32)
        eos = np.full(n, np.nan, dtype=np.float32)
        sos[valid_rc] = sos_band[rows[valid_rc], cols[valid_rc]]
        eos[valid_rc] = eos_band[rows[valid_rc], cols[valid_rc]]
        sos[~np.isfinite(sos)] = np.nan
        eos[~np.isfinite(eos)] = np.nan
        df[f"sos_{year}"] = sos
        df[f"eos_{year}"] = eos
    return df

def _load_annual_climate_for_coords(lons, lats, years):
        years = set(int(y) for y in years)
    coords = pd.DataFrame({"longitude": lons, "latitude": lats})
    coords["_lo"] = np.round(coords["longitude"], 5)
    coords["_la"] = np.round(coords["latitude"], 5)
    key = coords[["_lo", "_la"]].drop_duplicates()

    def load_chunked(paths, prefix):
        hits = []
        for fp in paths:
            cols = pd.read_csv(fp, nrows=0).columns
            ycols = [c for c in cols if c.startswith(prefix) and int(c.split("_")[-1]) in years]
            if not ycols:
                continue
            usecols = ["longitude", "latitude"] + ycols
            for chunk in pd.read_csv(fp, usecols=usecols, chunksize=250_000):
                chunk = chunk.assign(
                    _lo=np.round(chunk["longitude"].to_numpy(), 5),
                    _la=np.round(chunk["latitude"].to_numpy(), 5),
                )
                sub = chunk.merge(key, on=["_lo", "_la"], how="inner")
                if len(sub):
                    hits.append(sub.drop(columns=["longitude", "latitude"]))
        if not hits:
            return coords[["_lo", "_la"]].copy()
        df = pd.concat(hits, ignore_index=True)
        ycols = [c for c in df.columns if c.startswith(prefix)]
        return df.groupby(["_lo", "_la"], as_index=False)[ycols].first()

    t_paths = [
        "../../data/climate_data/tables/climate_data/temp/temp-1982-1999.csv",
        "../../data/climate_data/tables/climate_data/temp/temp-2000-2024.csv",
    ]
    p_paths = [
        "../../data/climate_data/tables/climate_data/prcp/prcp-1982-1999.csv",
        "../../data/climate_data/tables/climate_data/prcp/prcp-2000-2024.csv",
    ]
    print("  loading annual T from climate tables...", flush=True)
    tdf = load_chunked(t_paths, "annual_t_")
    print("  loading annual P from climate tables...", flush=True)
    pdf = load_chunked(p_paths, "annual_p_")
    out = coords.copy()
    out = out.merge(tdf, on=["_lo", "_la"], how="left")
    out = out.merge(pdf, on=["_lo", "_la"], how="left")
    return out.drop(columns=["_lo", "_la"])

def read_satellite_data(veg_type, satellite):
    veg_class = pd.read_csv("../../data/veg_class_data/tables/veg_class.csv")
    clim_fp = f"../../data/satellite_data/tables/phenology_climate/{satellite}.csv"
    if satellite == "gimms" and not os.path.exists(clim_fp):
        print(f"  {clim_fp} missing — rebuilding annual T/P from climate tables", flush=True)
        forest = veg_class[veg_class["veg_class"].isin([11, 12, 13, 14])].copy()
        if veg_type in (11, 12, 13, 14):
            forest = forest[forest["veg_class"] == veg_type].copy()
        years = list(range(1982, 2023))
        df_satellite = _load_annual_climate_for_coords(
            forest["longitude"].values, forest["latitude"].values, years
        )
        df = forest.merge(df_satellite, on=["longitude", "latitude"], how="inner")
    else:
        df_satellite = pd.read_csv(clim_fp)
        df = pd.merge(df_satellite, veg_class, on=["latitude", "longitude"], how="inner")
    if veg_type in (11, 12, 13, 14):
        df = df[df["veg_class"].isin([veg_type])]
    else:
        df = df[df["veg_class"].isin([11, 12, 13, 14])]

    eos_cols = [col for col in df.columns if "eos" in col]
    t_cols = [col for col in df.columns if "annual_t" in col]
    p_cols = [col for col in df.columns if "annual_p" in col]
    sos_cols = [col for col in df.columns if "sos" in col]

    if satellite == "gimms":
        years = [str(y) for y in range(1982, 2023)]
        df = df.drop(columns=[c for c in eos_cols + sos_cols], errors="ignore")
        df = load_gimms_snowfilter_sos_eos(df, [int(y) for y in years])
        eos_cols = [col for col in df.columns if col.startswith("eos_")]
        sos_cols = [col for col in df.columns if col.startswith("sos_")]
    elif satellite == "avhrr":
        years = [str(y) for y in range(1982, 2017)]
    elif satellite == "modis":
        years = [str(y) for y in range(2001, 2024)]
        mask_sos = (df[sos_cols] < 0).any(axis=1)
        mask_eos = (df[eos_cols] > 365).any(axis=1)
        df = df[~(mask_sos | mask_eos)].copy()
    else:
        years = [str(y) for y in range(2013, 2023)]
        mask_sos = (df[sos_cols] < 0).any(axis=1)
        mask_eos = (df[eos_cols] > 365).any(axis=1)
        df = df[~(mask_sos | mask_eos)].copy()

    cols = years
    df = df[[col for col in eos_cols + t_cols + p_cols + sos_cols if any(y in col for y in cols)] + ["latitude", "longitude", "veg_class"]].copy()
    t_cols_df = [col for col in df.columns if "annual_t" in col]
    df[t_cols_df] = df[t_cols_df] - 273.5  # Convert temperature
    df.columns = df.columns.str.replace(r"\D*(\d{4})$", lambda m: f"{m.group(0)[0:-4]}{m.group(1)}", regex=True)
    df["annual_t"] = df[[col for col in df.columns if "annual_t" in col]].mean(axis=1)
    df["annual_p"] = df[[col for col in df.columns if "annual_p" in col]].mean(axis=1)
    eos_year_cols = [col for col in df.columns if col.startswith("eos_")]
    sos_year_cols = [col for col in df.columns if col.startswith("sos_")]
    df["eos"] = df[eos_year_cols].mean(axis=1)
    df["sos"] = df[sos_year_cols].mean(axis=1)
    if satellite == "gimms":
        df = df[df["eos"].notna()].copy()
    return df


In [ ]:
## Plot EOS turning point for each forest type
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

def plot_mean_eos_by_temp_precip(df, temp_col, precip_col,
                                 eos_col, precip_threshold):
    df = df.copy()

    df['precip_bin'] = np.where(df[precip_col] <= precip_threshold, 0, 1)
    bins = sorted(df['precip_bin'].dropna().unique())

    fig, ax = plt.subplots(figsize=(6, 4))

    for bin_val in bins:
        sub_df = df[df['precip_bin'] == bin_val].copy()
        if len(sub_df) < 5:
            continue

        sub_df['temp_1C_bin'] = np.floor(sub_df[temp_col] * 10) / 10
        counts = sub_df['temp_1C_bin'].value_counts()
        valid_bins = counts[counts >= 10].index
        filtered_df = sub_df[sub_df['temp_1C_bin'].isin(valid_bins)]
        if len(filtered_df) < 5:
            continue

        raw_x = filtered_df[temp_col].values
        raw_y = filtered_df[eos_col].values

        def _trim_mean_std(s, lo=2.0, hi=98.0):
            vals = s.dropna().to_numpy(dtype=float)
            if vals.size == 0:
                return np.nan, np.nan
            q_lo, q_hi = np.nanpercentile(vals, [lo, hi])
            trimmed = vals[(vals >= q_lo) & (vals <= q_hi)]
            if trimmed.size == 0:
                return np.nan, np.nan
            return float(np.mean(trimmed)), float(np.std(trimmed, ddof=1)) if trimmed.size > 1 else 0.0

        stats = (
            filtered_df.groupby('temp_1C_bin')[eos_col]
            .apply(lambda s: pd.Series(_trim_mean_std(s), index=['mean', 'std']))
            .unstack()
        )
        x = stats.index.to_numpy(dtype=float)
        y = stats['mean'].to_numpy(dtype=float)
        yerr = stats['std'].to_numpy(dtype=float)
        ok = np.isfinite(x) & np.isfinite(y)
        x, y, yerr = x[ok], y[ok], yerr[ok]

        if len(x) < 5:
            continue

        def format_p(p):
            if p < 0.01:
                return "p<0.01"
            elif p < 0.05:
                return "p<0.05"
            else:
                return f"p={p:.2f}"

        if bin_val == 0:
            ax.plot(x, y, color='#e03c31', lw=2, label='Mean EOS (red)')
            ax.fill_between(x, y - yerr, y + yerr, color='#e03c31', alpha=0.2, label='±1 Std. Dev. (red)')

            peak_candidates = (x >= 6.0) & (x <= 8.0)
            if not np.any(peak_candidates):
                continue
            peak_idx = np.argmax(y[peak_candidates])
            peak_x_vals = x[peak_candidates]
            bp = peak_x_vals[peak_idx]  # Breakpoint

            ax.set_ylim(240, 320)
            ax.axvline(x=bp, color='black', linestyle='--', label='Breakpoint (peak)')
            ax.text(bp - 0.5, 300, f'{bp:.2f}°C', color='black', fontsize=12, ha='center')

            left_mask = raw_x <= bp
            right_mask = raw_x > bp

            if np.sum(left_mask) >= 2 and np.sum(right_mask) >= 2:
                slope1, intercept1, r_value1, p_value1, std_err1 = linregress(raw_x[left_mask], raw_y[left_mask])
                slope2, intercept2, r_value2, p_value2, std_err2 = linregress(raw_x[right_mask], raw_y[right_mask])

                x_left = np.linspace(raw_x[left_mask].min(), raw_x[left_mask].max(), 50)
                y_left = slope1 * x_left + intercept1
                ax.plot(x_left, y_left, color='black', linestyle='--',
                        label=f'Left slope={slope1:.2f}, p={p_value1:.2f}')

                x_right = np.linspace(raw_x[right_mask].min(), raw_x[right_mask].max(), 50)
                y_right = slope2 * x_right + intercept2
                ax.plot(x_right, y_right, color='black', linestyle='--',
                        label=f'Right slope={slope2:.2f}, p={p_value2:.2f}')

                ax.text(0.02, 0.02, f'Slope={slope1:.2f}\nR={r_value1:.2f}\n{format_p(p_value1)}',
                        transform=ax.transAxes, fontsize=10, color='black', ha='left', va='bottom')
                ax.text(0.98, 0.02, f'Slope={slope2:.2f}\nR={r_value2:.2f}\n{format_p(p_value2)}',
                        transform=ax.transAxes, fontsize=10, color='black', ha='right', va='bottom')
            else:
                slope, intercept, r_value, p_value, std_err = linregress(raw_x, raw_y)
                x_all = np.linspace(raw_x.min(), raw_x.max(), 100)
                y_all = slope * x_all + intercept
                ax.plot(x_all, y_all, color='black', linestyle='--',
                        label=f'Slope={slope:.2f}, p={p_value:.2f}')
                ax.text(0.02, 0.02, f'Slope={slope:.2f}\nR={r_value:.2f}\n{format_p(p_value)}',
                        transform=ax.transAxes, fontsize=10, color='black', ha='left', va='bottom')
        else:
            ax.plot(x, y, color='gray', lw=2, label='Mean EOS (gray)')
            ax.fill_between(x, y - yerr, y + yerr, color='gray', alpha=0.2, label='±1 Std. Dev. (gray)')

    ax.set_ylim(240, 340)
    ax.set_xlabel("MAT (°C)", fontsize=14)
    ax.set_ylabel("EOS (DOY)", fontsize=14)
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    ax.set_xticks([0, 10, 20])
    ax.set_yticks([240, 260, 280, 300, 320, 340])
    from matplotlib.lines import Line2D
    custom_legend = [
        Line2D([0], [0], color='gray', lw=5, label='Wet regions'),
        Line2D([0], [0], color='#e03c31', lw=5, label='Dry regions'),
    ]
    leg = plt.legend(
        handles=custom_legend,
        fontsize=10,
        frameon=False,
        loc='upper left',
        handlelength=1.0,
        handleheight=0.8
    )
    for line in leg.get_lines():
        line.set_linewidth(7)
    plt.tight_layout()
    return fig


Plot DB / MF


In [ ]:
satellite = 'gimms'

min_z_count = 0
veg_type = 0
df = read_satellite_data(veg_type, satellite)
df = df[
    (df['annual_t'] >= -20) & (df['annual_t'] <= 20) &
    (df['annual_p'] >= 0) & (df['annual_p'] <= 4)
]

# P thresholds from right-side EOS slope switch (decline → rise; see test chunk)
# EN: 0.7 m; DB: 1.0 m; MF: 1.1 m
fig2 = plot_mean_eos_by_temp_precip(df[df['veg_class'] == 13], 'annual_t', 'annual_p', 'eos', 1.0)
fig3 = plot_mean_eos_by_temp_precip(df[df['veg_class'] == 14], 'annual_t', 'annual_p', 'eos', 1.1)

# fig1.savefig(f"../../results/ed_figures/ed_fig3/eos_EN.png", dpi=500, bbox_inches='tight')
fig2.savefig(f"../../results/ed_figures/ed_fig3/eos_DB.png", dpi=500, bbox_inches='tight')
fig3.savefig(f"../../results/ed_figures/ed_fig3/eos_MF.png", dpi=500, bbox_inches='tight')
